##存取Google雲端硬碟##

In [ ]:
!nvidia-smi  # 顯示GPU的資訊

Sun May 11 09:27:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 解壓縮車牌資料檔

In [ ]:
!scp '/content/drive/MyDrive/license_plate.zip' '/content/data.zip'
!unzip '/content/data.zip' -d '/content/license_plate'

Streaming output truncated to the last 5000 lines.
 extracting: /content/license_plate/train/labels/002572_jpg.rf.98cf34cb87fb5e642a0cfc5dddc420f7.txt  
 extracting: /content/license_plate/train/labels/002572_jpg.rf.9b8b30172e27cc2974628d216a793d56.txt  
 extracting: /content/license_plate/train/labels/002572_jpg.rf.c306e8875cac908515900681458384ab.txt  
 extracting: /content/license_plate/train/labels/002572_jpg.rf.cc13bc09884688b86db6fb0022a2e153.txt  
 extracting: /content/license_plate/train/labels/002572_jpg.rf.de56acb61b4e03806955b3fcfa9e02be.txt  
 extracting: /content/license_plate/train/labels/002572_jpg.rf.f3e0e606aee711fbf83618ee429b8ed4.txt  
 extracting: /content/license_plate/train/labels/002573_jpg.rf.02997cf36ada56e04a53fb8210ba5f03.txt  
 extracting: /content/license_plate/train/labels/002573_jpg.rf.2ac2697f19d06a7726f7e30e30a9b2be.txt  
 extracting: /content/license_plate/train/labels/002573_jpg.rf.5811498011ce57fa021358636b0f6133.txt  
 extracting: /content/license_p

## 安裝YOLO並載入預設的模型

In [ ]:
!pip install ultralytics  # 安裝YOLO

import ultralytics
ultralytics.checks()

from ultralytics import YOLO
model = YOLO("yolo12n.pt")  # 下載YOLO模型

Ultralytics 8.3.131 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.4/112.6 GB disk)


100%|██████████| 5.34M/5.34M [00:00<00:00, 390MB/s]


## 進行遷移學習並儲存訓練完成的模型和數據

In [ ]:
model.train(
    data="/content/license_plate/data.yaml",  # YAML檔的路徑
    epochs=30,  # 訓練次數
    batch=16,
    imgsz=640,  # 訓練的影像大小
)

# 複製資料夾的命令
!scp -r /content/runs /content/drive/MyDrive/YOLO_runs

# 複製資料夾的Python程式
# import shutil

# src_path = '/content/runs'  # 來源路徑
# dest_path = '/content/drive/MyDrive/YOLO_runs' # 目標路徑
# shutil.copytree(src_path, dest_path)  # 開始複製資料夾

Ultralytics 8.3.121 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolo12n.pt, data=/content/license_plate/data.yaml, epochs=20, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_

100%|██████████| 755k/755k [00:00<00:00, 164MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  2    180864  ultralytics.nn.modules.block.A2C2f           [128, 128, 2, True, 4]        
  7                  -1  1    295424  ultralytics

 21        [14, 17, 20]  1    430867  ultralytics.nn.modules.head.Detect           [1, [64, 128, 256]]           
YOLOv12n summary: 272 layers, 2,568,243 parameters, 2,568,227 gradients, 6.5 GFLOPs

Transferred 640/691 items from pretrained weights
Freezing layer 'model.21.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


100%|██████████| 5.35M/5.35M [00:00<00:00, 241MB/s]


...

                   all        497        633      0.985      0.981      0.993      0.898



20 epochs completed in 2.217 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train/weights/best.pt, 5.5MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.121 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:07<00:00,  2.07it/s]


                   all        497        633      0.985      0.981      0.993      0.898
Speed: 0.3ms preprocess, 4.0ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to runs/detect/train


'/content/drive/MyDrive/YOLO_runs'